<a href="https://colab.research.google.com/github/myanh-tran/BananaHack-2026---cat-steals-socks/blob/ngu/Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Test

In [8]:
!pip install paddlepaddle

  Using cached paddlepaddle-3.3.1-cp313-cp313-manylinux1_x86_64.whl.metadata (8.8 kB)
Using cached paddlepaddle-3.3.1-cp313-cp313-manylinux1_x86_64.whl (194.8 MB)


In [2]:
# 1. Tải bộ mã nguồn gốc (như đã làm trên Kaggle)
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

# 2. Di chuyển vào thư mục
%cd PaddleOCR

# 3. Chạy lệnh nhận diện bằng script chuẩn của hãng (bỏ qua vỏ bọc bị lỗi)


Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353900, done.
remote: Counting objects: 100% (1212/1212), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 353900 (delta 1126), reused 1038 (delta 1038), pack-reused 352688 (from 2)
Receiving objects: 100% (353900/353900), 1.87 GiB | 19.95 MiB/s, done.
Resolving deltas: 100% (279888/279888), done.
/content/PaddleOCR
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/content/PaddleOCR/ppocr/postprocess/rec_postprocess.py:1239: SyntaxWarning: invalid escape sequence '\W'
  noletter = "[\W_^\d]"
/content/PaddleOCR/ppocr/postprocess/rec_postprocess.py:1479: SyntaxWarning: invalid escape sequence '\W'
  noletter = "[\W_^\d]"
/content/PaddleOCR/p

In [5]:
!python tools/infer/predict_rec.py \
    --image_dir="/content/oto.png" \
    --rec_model_dir="/content/inference_model" \
    --rec_char_dict_path="/content/dict.txt" \
    --use_gpu=False

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/10/11 11:03:37] ppocr INFO: In PP-OCRv3, rec_image_shape parameter defaults to '3, 48, 320', if you are using recognition model with PP-OCRv2 or an older version, please set --rec_image_shape='3,32,320
[2026/10/11 11:03:37] ppocr INFO: Predicts of /content/oto.png:('30MF45678', 0.9135012030601501)


In [8]:
!pip install -q paddlepaddle paddleocr ultralytics opencv-python-headless


In [9]:
%cd /content
!ls inference_model

/content
inference.json	inference.pdiparams  inference.yml


In [10]:
import cv2
from plate_ocr import PlateOCR
ocr = PlateOCR("inference_model")
img = cv2.imread("/content/oto.png")      # ảnh biển đã cắt sát
print(ocr.read(img))

('30MF45678', 0.9366124272346497)


In [12]:
from ultralytics import YOLO
det = YOLO("best.pt")
img = cv2.imread("/content/Screenshot 2026-10-10 165719.png")
r = det(img)[0]
for (x1, y1, x2, y2), conf in zip(r.boxes.xyxy.cpu().numpy().astype(int), r.boxes.conf):
    crop = img[y1:y2, x1:x2]
    print(round(float(conf), 2), ocr.read(crop))


0: 256x640 1 License_Plate, 7.9ms
Speed: 1.5ms preprocess, 7.9ms inference, 1.4ms postprocess per image at shape (1, 3, 256, 640)
0.72 ('30M45678', 0.9671866297721863)


In [13]:
from difflib import SequenceMatcher

def match_plate(text, db_plates, min_ratio=0.8, margin=0.1):
    scored = sorted(((SequenceMatcher(None, text, p).ratio(), p) for p in db_plates), reverse=True)
    if not scored or scored[0][0] < min_ratio:
        return None                      # không đủ giống
    if len(scored) > 1 and scored[0][0] - scored[1][0] < margin:
        return None                      # nhập nhằng giữa 2 biển, cần người xác nhận
    return scored[0][1]

In [14]:
# """
# Test ca luong: (YOLO cat bien) -> OCR -> so khop DB, in bang ket qua + ti le dung.

# Vi du (anh la anh xe nguyen khung):
#   python test_pipeline.py --images test_xe --truth truth.txt --yolo best.pt --model inference_model

# Vi du (anh da la anh bien cat san, bo qua YOLO):
#   python test_pipeline.py --images test_bien --truth truth.txt --model inference_model

# truth.txt: moi dong "ten_anh.jpg BIENSO" (bien so that cua anh do), giong labels.txt
# db.txt   : moi dong 1 bien so co trong CSDL (neu bo trong thi lay tu truth.txt)
# """
# import argparse, os, csv
# from difflib import SequenceMatcher
# import cv2
# from plate_ocr import PlateOCR


# def match_plate(text, db, min_ratio=0.8, margin=0.1):
#     scored = sorted(((SequenceMatcher(None, text, p).ratio(), p) for p in db), reverse=True)
#     if not scored or scored[0][0] < min_ratio:
#         return None
#     if len(scored) > 1 and scored[0][0] - scored[1][0] < margin:
#         return None
#     return scored[0][1]


# def read_pairs(path):
#     d = {}
#     for line in open(path, encoding="utf-8"):
#         p = line.split()
#         if len(p) == 2:
#             d[p[0]] = p[1]
#     return d


# def main():
#     ap = argparse.ArgumentParser()
#     ap.add_argument("--images", required=True)
#     ap.add_argument("--truth", required=True)
#     ap.add_argument("--model", required=True)
#     ap.add_argument("--yolo", default=None)
#     ap.add_argument("--db", default=None)
#     ap.add_argument("--out", default="ket_qua_test.csv")
#     a = ap.parse_args()

#     truth = read_pairs(a.truth)
#     db = sorted(set(l.strip() for l in open(a.db) if l.strip())) if a.db else sorted(set(truth.values()))
#     ocr = PlateOCR(a.model)
#     det = None
#     if a.yolo:
#         from ultralytics import YOLO
#         det = YOLO(a.yolo)

#     rows, n, exact, ok_match, wrong_match, unknown = [], 0, 0, 0, 0, 0
#     for name, gt in truth.items():
#         path = os.path.join(a.images, name)
#         img = cv2.imread(path)
#         if img is None:
#             print("khong doc duoc:", name)
#             continue
#         if det is not None:
#             r = det(img, verbose=False)[0]
#             if len(r.boxes) == 0:
#                 rows.append([name, gt, "", "", "KHONG THAY BIEN"])
#                 n += 1
#                 unknown += 1
#                 continue
#             i = int(r.boxes.conf.argmax())
#             x1, y1, x2, y2 = r.boxes.xyxy[i].cpu().numpy().astype(int)
#             img = img[y1:y2, x1:x2]
#         text, score = ocr.read(img)
#         m = match_plate(text, db)
#         n += 1
#         exact += text == gt
#         if m is None:
#             unknown += 1
#             status = "KHONG KHOP DB"
#         elif m == gt:
#             ok_match += 1
#             status = "DUNG"
#         else:
#             wrong_match += 1
#             status = "NHAM XE KHAC"
#         rows.append([name, gt, text, m or "", status])

#     with open(a.out, "w", newline="", encoding="utf-8") as f:
#         w = csv.writer(f)
#         w.writerow(["anh", "that", "doc_duoc", "khop_db", "ket_qua"])
#         w.writerows(rows)

#     print(f"\nTong {n} anh")
#     print(f"  OCR doc dung nguyen chuoi : {exact}/{n} = {exact / max(n, 1):.0%}")
#     print(f"  Nhan dung xe (sau khop DB): {ok_match}/{n} = {ok_match / max(n, 1):.0%}")
#     print(f"  Khong khop / can xac nhan : {unknown}/{n}")
#     print(f"  NHAM sang xe khac (nguy hiem): {wrong_match}/{n}")
#     print("Chi tiet:", a.out)
#     print("\nCac anh sai:")
#     for r in rows:
#         if r[4] != "DUNG":
#             print("  ", r)


# if __name__ == "__main__":
#     main()

usage: colab_kernel_launcher.py [-h] --images IMAGES --truth TRUTH
                                --model MODEL [--yolo YOLO] [--db DB]
                                [--out OUT]
colab_kernel_launcher.py: error: the following arguments are required: --images, --truth, --model
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.13/argparse.py", line 1965, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/argparse.py", line 2262, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
               ', '.join(required_actions))
argparse.ArgumentError: the following arguments are required: --images, --truth, --model

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_13369/2619307643.py", line 105, in <cell line: 0>
    main()
    ~~~~^^
  File "/tmp/ipykernel_13369/2619307643.py"

TypeError: object of type 'NoneType' has no len()

In [16]:
%cd /content
!ls
!python test_pipeline.py --images test_xe --truth truth.txt --yolo best.pt --model inference_model --db db.txt

/content
best.pt   inference_model   PaddleOCR	  sample_data	    test_xe
db.txt	  ket_qua_test.csv  plate_ocr.py  test_pipeline.py  tot.png
dict.txt  oto.png	    __pycache__   test.png	    truth.txt
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)

Tong 4 anh
  OCR doc dung nguyen chuoi : 4/4 = 100%
  Nhan dung xe (sau khop DB): 4/4 = 100%
  Khong khop / can xac nhan : 0/4
  NHAM sang xe khac (nguy hiem): 0/4
Chi tiet: ket_qua_test.csv

Cac anh sai:
